In [1]:
!pip -q install -U timm
import os, gc, math, time, glob, numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F, timm
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

DEV = "cuda"; SZ = 448
COMP = "/kaggle/input/competitions/ai-of-god-5/participant_data"; OUT = "/kaggle/working"

RUN = "EVA448"                 # then rerun everything with "EVA448"
CFG = {"D3L448": dict(ARCH="vit_large_patch16_dinov3.lvd1689m", NORM="imagenet"),
       "EVA448": dict(ARCH="eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", NORM="clip")}
ARCH, TAG = CFG[RUN]["ARCH"], RUN
print(timm.list_models("eva02_large*448*"))            # if the EVA name above fails, pick one from this list
EPOCHS, BS, N_UNFROZEN = 20, 8, 6                      # BS=4 if you hit out-of-memory
LR_TOP, LLD, LR_HEAD, WD, EMA_D = 3e-5, 0.85, 1e-3, 0.05, 0.99
RUN_FOLDS = [0, 1]

NORMS = {"imagenet": ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
         "clip": ([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])}
MEAN = torch.tensor(NORMS[CFG[RUN]["NORM"]][0], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor(NORMS[CFG[RUN]["NORM"]][1], device=DEV).view(1, 3, 1, 1)
PERMS = [[0, 1, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 67.0 MB/s eta 0:00:00
['eva02_large_patch14_448']


In [2]:
df = pd.read_csv(f"{COMP}/train.csv"); te = pd.read_csv(f"{COMP}/sample_submission.csv")
le = LabelEncoder(); y = le.fit_transform(df.animal_class); NC = len(le.classes_)

def resolve(d, n):
    p = f"{COMP}/{d}/{n}"
    if os.path.exists(p): return p
    for e in (".png", ".jpg", ".jpeg"):
        if os.path.exists(p + e): return p + e
    raise FileNotFoundError(p)

def cache(paths, name):
    f = f"{OUT}/{name}_{SZ}.npy"
    if not os.path.exists(f):
        arr = np.zeros((len(paths), SZ, SZ, 3), np.uint8)
        for i, p in enumerate(tqdm(paths)):
            im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]; s = SZ / max(h, w)
            im = cv2.resize(im, (max(1, round(w*s)), max(1, round(h*s))), interpolation=cv2.INTER_AREA)
            oy, ox = (SZ - im.shape[0])//2, (SZ - im.shape[1])//2
            arr[i, oy:oy+im.shape[0], ox:ox+im.shape[1]] = im
        np.save(f, arr); del arr
    return np.load(f, mmap_mode="r")

Xtr = cache([resolve("train_images", n) for n in df.image_id], "train")
Xte = cache([resolve("test_images", n) for n in te.image_id], "test")
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(y)), y))
print(Xtr.shape, Xte.shape, NC)

  0%|          | 0/3087 [00:00<?, ?it/s]

  0%|          | 0/1179 [00:00<?, ?it/s]

(3087, 448, 448, 3) (1179, 448, 448, 3) 72


In [3]:
# ---------- GPU augmentation (color destroyed, shape kept) ----------
def curves(x, K=9):
    B, C, H, W = x.shape
    k = torch.cumsum(torch.rand(B,3,K-1, device=DEV)**2 + 1e-3, -1); k = k / k[..., -1:]
    k = torch.cat([torch.zeros(B,3,1, device=DEV), k], -1).reshape(B*C, K)
    t = x.clamp(0,1).reshape(B*C, -1) * (K-1); lo = t.floor().clamp(max=K-2).long(); fr = t - lo
    return (torch.gather(k,1,lo)*(1-fr) + torch.gather(k,1,lo+1)*fr).view(B,C,H,W)

def color_aug(x):
    B = x.shape[0]; r = torch.rand(B,1,1,1, device=DEV)
    M = torch.rand(B,3,3, device=DEV)**2 + 0.02; M = M / M.sum(-1, keepdim=True)
    xm = torch.einsum('bij,bjhw->bihw', M, x)
    lo, hi = xm.amin((2,3), keepdim=True), xm.amax((2,3), keepdim=True)
    xm = curves((xm-lo)/(hi-lo+1e-3))
    xs = torch.gather(x, 1, torch.argsort(torch.rand(B,3, device=DEV),1)[:,:,None,None].expand_as(x))
    xg = x.mean(1, keepdim=True).expand_as(x)
    out = torch.where(r<0.6, xm, torch.where(r<0.8, xs, xg))
    return torch.where(torch.rand(B,1,1,1, device=DEV)<0.15, 1-out, out)

def degrade(x):
    B = x.shape[0]; w = torch.ones(3,1,5,5, device=DEV)/25
    xb = F.conv2d(F.pad(x,(2,2,2,2),mode='reflect'), w, groups=3)
    x = torch.where(torch.rand(B,1,1,1, device=DEV)<0.3, xb, x)
    lv = 2**torch.randint(3,6,(B,1,1,1), device=DEV).float() - 1
    x = torch.where(torch.rand(B,1,1,1, device=DEV)<0.2, torch.round(x*lv)/lv, x)
    x = x + (torch.rand(B,1,1,1, device=DEV)<0.5) * torch.rand(B,1,1,1, device=DEV)*0.05 * torch.randn_like(x)
    return x.clamp(0,1)

def geom(x):
    B = x.shape[0]; u = lambda: torch.rand(B, device=DEV)*2-1
    ang = u()*math.radians(20); sc = 0.75 + torch.rand(B, device=DEV)*0.5; asp = torch.exp(u()*0.1)
    flip = (torch.rand(B, device=DEV)<0.5).float()*2-1; c, s = torch.cos(ang), torch.sin(ang)
    sx, sy = 1/(sc*asp), asp/sc; th = torch.zeros(B,2,3, device=DEV)
    th[:,0,0]=c*sx*flip; th[:,0,1]=-s*sy; th[:,0,2]=u()*0.1
    th[:,1,0]=s*sx*flip; th[:,1,1]= c*sy; th[:,1,2]=u()*0.1
    return F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode='zeros', align_corners=False)

def cutout(x, p=0.3, mf=0.2):
    B,_,H,W = x.shape
    for i in torch.nonzero(torch.rand(B)<p).flatten().tolist():
        h, w = int(H*mf*np.random.uniform(.3,1)), int(W*mf*np.random.uniform(.3,1))
        y0, x0 = np.random.randint(0,H-h), np.random.randint(0,W-w)
        x[i,:,y0:y0+h,x0:x0+w] = torch.rand(3,1,1, device=DEV)
    return x

def train_aug(xb):
    x = xb.to(DEV).permute(0,3,1,2).float()/255
    rows = (x.amax((1,3)) > 0); cols = (x.amax((1,2)) > 0)            # non-padded bounding box
    box = (rows[:,:,None] & cols[:,None,:])[:,None].float()
    x = degrade(color_aug(x)) * box                                     # keep letterbox black like test
    return (cutout(geom(x)) - MEAN) / STD

# ---------- model ----------
def build():
    m = timm.create_model(ARCH, pretrained=True, num_classes=NC, img_size=SZ, drop_rate=0.3)
    for p in m.parameters(): p.requires_grad = False
    groups, nb = [], len(m.blocks)
    def add(params, lr):
        params = list(params)
        for p in params: p.requires_grad = True
        groups.append(dict(params=[p for p in params if p.ndim>=2], lr=lr, weight_decay=WD))
        groups.append(dict(params=[p for p in params if p.ndim<2],  lr=lr, weight_decay=0.0))
    for i in range(nb-N_UNFROZEN, nb): add(m.blocks[i].parameters(), LR_TOP * LLD**(nb-1-i))
    for nm in ("norm", "fc_norm"):
        mod = getattr(m, nm, None)
        if mod is not None and len(list(mod.parameters())): add(mod.parameters(), LR_TOP)
    add(m.head.parameters(), LR_HEAD)
    return m, [g for g in groups if g['params']]

class EMA:
    def __init__(s, m): s.n = 0; s.sh = {k: p.detach().clone() for k,p in m.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(s, m):
        s.n += 1; d = min(EMA_D, (1+s.n)/(10+s.n))
        for k,p in m.named_parameters():
            if k in s.sh: s.sh[k].mul_(d).add_(p.detach(), alpha=1-d)
    @torch.no_grad()
    def apply(s, m):
        for k,p in m.named_parameters():
            if k in s.sh: p.copy_(s.sh[k])

@torch.no_grad()
def predict(m, X, idx, bs=16):                       # 4 channel perms x flip = 8 passes
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        x = torch.from_numpy(np.asarray(X[idx[i:i+bs]])).to(DEV).permute(0,3,1,2).float()/255; acc = 0
        for p in PERMS:
            for f in (0,1):
                v = x[:, p]; v = v.flip(3) if f else v
                with torch.autocast('cuda', dtype=torch.float16): acc = acc + m((v-MEAN)/STD).float().softmax(1)
        out.append((acc/8).cpu())
    return torch.cat(out).numpy()

# ---------- one fold ----------
def run_fold(k):
    fo, fi, ft = (f"{OUT}/{n}_{TAG}_f{k}.npy" for n in ("oof","va_idx","test"))
    if all(os.path.exists(f) for f in (fo,fi,ft)): print(f"fold {k} done already"); return
    tr, va = folds[k]; torch.manual_seed(k); np.random.seed(k)
    m, groups = build(); m.to(DEV); trainable = [p for g in groups for p in g['params']]
    opt = torch.optim.AdamW(groups); spe = len(tr)//BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in groups], total_steps=EPOCHS*spe,
                                                pct_start=0.1, div_factor=25, final_div_factor=100)
    scaler, ema = torch.amp.GradScaler(), EMA(m)
    cnt = np.bincount(y[tr], minlength=NC)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(len(tr)/(NC*cnt), dtype=torch.float32, device=DEV), label_smoothing=0.1)
    t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); perm = np.random.permutation(tr); run = 0
        for b in range(spe):
            ii = np.sort(perm[b*BS:(b+1)*BS])
            x = train_aug(torch.from_numpy(np.asarray(Xtr[ii]))); yt = torch.from_numpy(y[ii]).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(m(x).float(), yt)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); scaler.step(opt); scaler.update(); sched.step(); ema.update(m)
            run += loss.item()
        print(f"fold {k} ep {ep+1}/{EPOCHS} loss {run/spe:.4f}  {(time.time()-t0)/60:.1f} min")
    ema.apply(m)                                                    # final weights = EMA
    p_va = predict(m, Xtr, va); p_te = predict(m, Xte, np.arange(len(te)))
    np.save(fo, p_va.astype(np.float32)); np.save(fi, va); np.save(ft, p_te.astype(np.float32))
    print(f"fold {k}: acc {np.mean(p_va.argmax(1)==y[va]):.4f}  macroF1 {f1_score(y[va], p_va.argmax(1), average='macro'):.4f}")
    del m, opt, ema, scaler, sched; gc.collect(); torch.cuda.empty_cache()

# ---------- run the chosen folds (time-guarded) ----------
T0, BUDGET_H, FOLD_H = time.time(), 10, 1.5        # FOLD_H = your measured hours per fold at this resolution
for k in RUN_FOLDS:
    if (time.time()-T0)/3600 + FOLD_H > BUDGET_H: print(f"stopping before fold {k}: not enough time"); break
    run_fold(k)

# ---------- OOF score on the finished folds (no submission from a partial run) ----------
oof = np.full((len(y), NC), np.nan, np.float32); tests = []
for k in range(5):
    if os.path.exists(f"{OUT}/oof_{TAG}_f{k}.npy"):
        oof[np.load(f"{OUT}/va_idx_{TAG}_f{k}.npy")] = np.load(f"{OUT}/oof_{TAG}_f{k}.npy"); tests.append(np.load(f"{OUT}/test_{TAG}_f{k}.npy"))
ok = ~np.isnan(oof[:,0])
print(f"folds done: {len(tests)} | rows {ok.sum()} | OOF macro F1 {f1_score(y[ok], oof[ok].argmax(1), average='macro'):.4f}")
if tests: np.save(f"{OUT}/test_{TAG}_mean.npy", np.mean(tests, 0))

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

fold 0 ep 1/20 loss 2.5145  4.8 min
fold 0 ep 2/20 loss 1.2466  9.6 min
fold 0 ep 3/20 loss 1.1036  14.4 min
fold 0 ep 4/20 loss 1.0061  19.1 min
fold 0 ep 5/20 loss 0.9611  23.9 min
fold 0 ep 6/20 loss 0.9071  28.7 min
fold 0 ep 7/20 loss 0.9172  33.5 min
fold 0 ep 8/20 loss 0.8931  38.2 min
fold 0 ep 9/20 loss 0.8763  43.0 min
fold 0 ep 10/20 loss 0.8586  47.8 min
fold 0 ep 11/20 loss 0.8188  52.6 min
fold 0 ep 12/20 loss 0.8260  57.3 min
fold 0 ep 13/20 loss 0.8415  62.1 min
fold 0 ep 14/20 loss 0.8243  66.9 min
fold 0 ep 15/20 loss 0.8286  71.7 min
fold 0 ep 16/20 loss 0.8237  76.4 min
fold 0 ep 17/20 loss 0.8269  81.2 min
fold 0 ep 18/20 loss 0.8219  86.0 min
fold 0 ep 19/20 loss 0.8163  90.8 min
fold 0 ep 20/20 loss 0.8110  95.5 min
fold 0: acc 0.9612  macroF1 0.9602
fold 1 ep 1/20 loss 2.5202  4.8 min
fold 1 ep 2/20 loss 1.2047  9.6 min
fold 1 ep 3/20 loss 1.1086  14.3 min
fold 1 ep 4/20 loss 1.0165  19.1 min
fold 1 ep 5/20 loss 0.9484  23.9 min
fold 1 ep 6/20 loss 0.9382  28.7 

In [4]:
PREV, NEWD = "/kaggle/input/datasets/kavyanshusingh/d3l336-dataset", "/kaggle/working/"     # or OUT for a run in this session
def load(tag, d):
    o = np.full((len(y), NC), np.nan, np.float32)
    for k in range(5):
        f = f"{d}/oof_{tag}_f{k}.npy"
        if os.path.exists(f): o[np.load(f"{d}/va_idx_{tag}_f{k}.npy")] = np.load(f)
    return o

A = load("D3L336", PREV); B = load(RUN, NEWD)
rows = np.where(~np.isnan(B[:, 0]) & ~np.isnan(A[:, 0]))[0]
print("rows compared:", len(rows))

def sc(P, r=rows): return accuracy_score(y[r], P[r].argmax(1)), f1_score(y[r], P[r].argmax(1), average="macro")
conf = (A[rows].max(1) + B[rows].max(1)) / 2                     # hard = least confident by both models
hard = rows[np.argsort(conf)[:int(0.2 * len(rows))]]
for name, P in [("D3L336", A), (RUN, B)]:
    a, f = sc(P); print(f"{name:<8} acc {a:.4f} F1 {f:.4f} | hard acc {sc(P, hard)[0]:.4f}")
for w in [.3, .5, .7]:
    P = (1 - w) * A + w * B; a, f = sc(P)
    print(f"blend w_new={w}: acc {a:.4f} F1 {f:.4f} | hard acc {sc(P, hard)[0]:.4f}")

ea, eb = A[rows].argmax(1) != y[rows], B[rows].argmax(1) != y[rows]
print(f"errors D3L336 {ea.sum()} | new {eb.sum()} | both wrong {(ea & eb).sum()} | "
      f"oracle (either right) acc {1 - (ea & eb).mean():.4f}")

rows compared: 1236
D3L336   acc 0.9871 F1 0.9868 | hard acc 0.9352
EVA448   acc 0.9644 F1 0.9638 | hard acc 0.8259
blend w_new=0.3: acc 0.9854 F1 0.9853 | hard acc 0.9271
blend w_new=0.5: acc 0.9830 F1 0.9826 | hard acc 0.9150
blend w_new=0.7: acc 0.9782 F1 0.9777 | hard acc 0.8947
errors D3L336 16 | new 44 | both wrong 10 | oracle (either right) acc 0.9919
